# fatAnalyze - single-case analysis

End-to-end pipeline for a single CT series. Edit the **Configuration** cell,
then Run-All.

Steps:
1. Load DICOM series + QC
2. TotalSegmentator segmentation (with disk cache)
3. ROI extraction (liver, pancreas, spleen in 3D; psoas at L3-approx)
4. HU histogram + range ratios
5. Clinical fat indicators
6. Visualization (per-ROI histograms, overlay, single-page summary)

In [3]:
# Configuration
from pathlib import Path

# --- Auto-detect project root from the fatanalyze/ package, regardless of
#     what CWD Jupyter was launched from. ---
def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "fatanalyze" / "__init__.py").exists():
            return p
    return start  # fall back to CWD

PROJECT_ROOT = _find_project_root(Path.cwd())
DICOM_DIR = PROJECT_ROOT / "data" / "my_case"   # <-- set to your DICOM folder
CACHE_DIR = PROJECT_ROOT / ".cache" / "totalseg_runs"
OUT_DIR   = PROJECT_ROOT / "fatanalyze-out"
DEVICE    = "cpu"                                # cpu | cuda | mps
FAST      = True                                 # TotalSegmentator fast mode

ROIS = [
    "liver",
    "pancreas",
    "spleen",
    "iliopsoas_left",
    "iliopsoas_right",
]

OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project: {PROJECT_ROOT}")
print(f"DICOM:   {DICOM_DIR}")
print(f"Cache:   {CACHE_DIR}")
print(f"Out:     {OUT_DIR}")

DICOM: data/my_case
Cache: /Users/mac/dev/fatAnalyze/notebooks/.cache/totalseg_runs
Out:   /Users/mac/dev/fatAnalyze/notebooks/fatanalyze-out


## 1. Load + QC

In [4]:
from fatanalyze.io.dicom_loader import load_ct_series

image, qc = load_ct_series(DICOM_DIR)
print(qc.summary())
if qc.warnings:
    print("Warnings:")
    for w in qc.warnings:
        print(f"  - {w}")
if qc.errors:
    raise SystemExit(f"QC failed: {qc.errors}")

FileNotFoundError: DICOM directory not found: data/my_case

## 2. TotalSegmentator segmentation

First run on a real CT takes 5-12 min on Apple Silicon (CPU + fast).
Subsequent runs hit the disk cache in <5s for unchanged volumes.

In [ ]:
import time
from fatanalyze.segment.totalseg import segment

t0 = time.time()
masks = segment(image, roi_names=ROIS, cache_dir=CACHE_DIR,
                device=DEVICE, fast=FAST)
print(f"Segmentation took {time.time()-t0:.1f}s")
for name, m in masks.items():
    arr_sum = int((sitk.GetArrayFromImage(m) > 0).sum()) if False else 0  # quick check
    print(f"  {name}: {m.GetSize()}")

## 3. ROI extraction

In [ ]:
import SimpleITK as sitk
from fatanalyze.roi.extractor import (
    get_3d_mask, find_l3_slice, get_l3_psoas_mask, extract_hu,
    mask_volume_ml, mask_area_cm2,
)

# 3D masks: liver, pancreas, spleen
liver = get_3d_mask(masks["liver"], keep_largest_cc=True, erode_voxels=0)
pancreas = get_3d_mask(masks["pancreas"], keep_largest_cc=True, erode_voxels=0)
spleen = get_3d_mask(masks["spleen"], keep_largest_cc=True, erode_voxels=0)

# L3-approx slice for psoas
z_center, z_min, z_max = find_l3_slice(
    masks["iliopsoas_left"], masks["iliopsoas_right"], n_buffer=1,
)
print(f"L3-approx slice z = {z_center} (buffer [{z_min}, {z_max}])")
psoas = get_l3_psoas_mask(
    masks["iliopsoas_left"], masks["iliopsoas_right"],
    z_center, n_buffer=1,
)

# HU extraction
hu = {
    "liver": extract_hu(image, liver),
    "pancreas": extract_hu(image, pancreas),
    "spleen": extract_hu(image, spleen),
    "psoas_L3": extract_hu(image, psoas),
}
for k, v in hu.items():
    print(f"  {k}: n={v.size} mean={v.mean():.1f}")

# Volumes
print("\nVolumes (ml):")
for name, m in [("liver", liver), ("pancreas", pancreas), ("spleen", spleen)]:
    print(f"  {name}: {mask_volume_ml(m):.1f} ml")
print(f"  psoas_L3 area at z={z_center}: {mask_area_cm2(psoas, z_center):.2f} cm^2")

## 4. Histograms + range ratios

In [ ]:
from fatanalyze.analysis.histogram import compute_ratios

spacing = image.GetSpacing()
results = {}
for name, arr in hu.items():
    cfg_name = "iliopsoas_left" if name == "psoas_L3" else name
    results[name] = compute_ratios(arr, cfg_name, spacing_xyz=spacing)
    r = results[name]
    print(f"{name:10s}  n={r.n_voxels:>7d}  mean={r.mean_hu:>6.1f}  vol={r.volume_ml:>7.1f}ml")
    for k, v in r.ratios.items():
        print(f"             {k:>15s}: {v*100:>5.1f}%")
    if r.clinical_flags:
        for f in r.clinical_flags:
            print(f"             ! {f}")

## 5. Clinical fat indicators

In [ ]:
from fatanalyze.analysis.fat_metrics import (
    liver_spleen_ratio, pancreas_spleen_difference, psoas_imat_fraction,
)

clinical = {
    "liver_spleen_ratio": liver_spleen_ratio(results["liver"], results["spleen"]),
    "pancreas_spleen_diff": pancreas_spleen_difference(results["pancreas"], results["spleen"]),
    "psoas_myosteatosis": psoas_imat_fraction(results["psoas_L3"]),
}
for k, v in clinical.items():
    print(f"{k}: {v}")

## 6. Visualizations

In [ ]:
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
from fatanalyze.viz.histogram_plot import plot_histogram
from fatanalyze.viz.overlay import plot_overlay

# Per-ROI histograms
for name, arr in hu.items():
    cfg_name = "iliopsoas_left" if name == "psoas_L3" else name
    fig = plot_histogram(arr, target_name=cfg_name, title=f"{name} HU")
    plt.show()

In [ ]:
# Psoas overlay at L3-approx slice
fig = plot_overlay(image, psoas, slice_index=z_center,
                   title=f"Psoas at L3-approx (z={z_center})")
plt.show()

In [ ]:
# Single-page summary
from fatanalyze.viz.summary import plot_summary

masks_for_summary = {
    "liver": liver, "pancreas": pancreas, "spleen": spleen,
    "iliopsoas_left": psoas,
}
fig = plot_summary(
    image, hu, masks_for_summary, results,
    clinical=clinical, l3_slice_index=z_center,
)
fig.savefig(OUT_DIR / "summary.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Summary saved to {OUT_DIR / 'summary.png'}")

## 7. Export metrics (CSV / JSON)

In [ ]:
import json
import pandas as pd

# Flatten results
rows = []
for name, r in results.items():
    row = {
        "target": name,
        "n_voxels": r.n_voxels,
        "volume_ml": r.volume_ml,
        "mean_hu": r.mean_hu,
        "median_hu": r.median_hu,
        "std_hu": r.std_hu,
    }
    row.update({f"ratio_{k}": v for k, v in r.ratios.items()})
    rows.append(row)
df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / "metrics.csv", index=False)
print(df.to_string(index=False))

with open(OUT_DIR / "clinical.json", "w") as f:
    json.dump(clinical, f, indent=2, default=str)
print(f"\nWrote {OUT_DIR / 'metrics.csv'}")
print(f"Wrote {OUT_DIR / 'clinical.json'}")